## Healthy Blood Prediction

KATE expects your code to define variables with specific names that correspond to certain things we are interested in.

KATE will run your notebook from top to bottom and check the latest value of those variables, so make sure you don't overwrite them.

* Remember to uncomment the line assigning the variable to your answer and don't change the variable or function names.
* Use copies of the original or previous DataFrames to make sure you do not overwrite them by mistake.

You will find instructions below about how to define each variable.

Once you're happy with your code, upload your notebook to KATE to check your feedback.

This dataset was collected to help predict from a blood exam if a patient is healthy or has hepatitis C ([source](https://archive.ics.uci.edu/ml/datasets/HCV+data)). It contains the laboratory results from the blood examinations of patients and their diagnosis.

While diagnostic pathways are based on expert rules (if-then-else rules), machine learning algorithms can go beyond these methods, and learn predictions rules directly from the data.

Our goal in this exercise is to implement a logistic regression model for prediction and understand some of its properties. In the second part of this exercise you will implement all methods by yourself, **without using the sklearn library**.

For reference, you find the `sklearn` Logistic Regression at [sklearn-logreg](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).

## Preparing the dataset

First, load the dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv('data/blood.csv')
df.head()

The 'Category' column indicates the health status of the patient. 

In [3]:
df["Category"].value_counts()

#### 1. Create target variable

Create a binary variable called `healthy` that is one when the patient is a healthy (category `'0=Blood Donor'`), and zero otherwise. Inspect how many samples are healthy and how many are not, using the `value_counts()` method. 

Calculate the percentage of healthy patient in the dataset, and save it to the variable `perc_healthy`.

In [4]:
# Add your code below
df["healthy"] = (df["Category"] == '0=Blood Donor').astype(int)
df["healthy"].value_counts()

# Calculate percentage of healthy patients
perc_healthy = df["healthy"].mean()


Once you have calculated `perc_healthy`, uncomment the cell below and print it out:

In [5]:
perc_healthy

Note that the dataset is imbalanced, with much higher proportion of healthy samples. This imbalance makes the task harders, as there is a small number of samples from the unhealthy class to learn from.

#### Inspect the columns of the model with the 'info' method. 

In [6]:
df.info()


#### 2. Clean the dataset 

Notice in `.info()` that the Non-Null values indicate that the dataset has missing values. Print the number of missing values for each column.

In [7]:
print(df.isnull().sum())


We must make a decision about how to correct them. Possible solutions are imputation, deletion of the problematic variables (columns) or deletion of problematic samples (rows). As there is a small number of samples with missing values (and for the sake of time), we will drop them. 

Create a clean dataframe by removing samples that have non-valid values, and save it to `df_clean`. 

Turn the categorical variable `Sex` to a valid input representation. Use a binary representation: `True` if equal to `m`, `False` otherwise.

And drop any columns that won't be used in the prediction model. Use `.info()` again to check the new dataset is clean.

*Hint: use the `dropna()` and `drop` methods.*

In [8]:
# Add your code below
df_clean = df.dropna().copy()  # Remove rows with missing values
df_clean["Sex"] = (df_clean["Sex"] == 'm')  # Convert to binary: True for male, False for female
df_clean = df_clean.drop(columns=['Category'])  # Drop the original category column


Once you have calculated `df_clean`, uncomment and run the cell below:

In [9]:
df_clean.head(3)

## Single variable model

We will start by analyzing a single variable column, which makes it easier to visualize the main principles of Logistic Regression. We choose the feature `AST` arbitrarily.

#### 3. Visualize how the target `healthy` depends on the variable `AST` with a scatter plot.

Use the `df.plot.scatter()` method and pass `ax=ax`, and `c=k` (color black).

Note that the `fig` and `ax` variables can be reused in cells below to plot over this figure.

In [10]:
fig,ax = plt.subplots()
# Add your code below
df.plot.scatter(x='AST', y='healthy', ax=ax, c='k')


#### 4. Implement a function that creates and fits a logistic regression model given the input and output data. 

In [11]:
# Add your code below
def logreg(X, Y):
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X.reshape(-1, 1) if X.ndim == 1 else X, Y)
    return model


#### 5. Fit a model named `model_AST` that predict the status `healthy` from the variable `AST`. 

Note: for the test on KATE to pass, use the dataframe `df` (and not `df_clean`) to define the input features `X` and the labels `y` to train your model `model_AST`.

In [12]:
# Add your code below
X = df['AST'].dropna().values
y = df.loc[df['AST'].notna(), 'healthy'].values
model_AST = logreg(X, y)


#### 6. Calculate the prediction probability for each data point. 

Using `model_AST`, calculate the prediction probabilities for all data points, saved as `y_prob`. 

Plot the result using `ax.scatter()`, which will reuse the plot created above, and the command `fig` will plot the resulting figure here. Use the color blue for this new plot, using the argument `color='tab:blue'`. 

Note: the `tab` colors are the new standard matplotlib colors (the "Tableau Palette").

In [13]:
# Add your code below
# Y_prob = ...
# plt.scatter...

y_prob = model_AST.predict_proba(X.reshape(-1, 1))[:, 1]
ax.scatter(X, y_prob, color='tab:blue')
fig

#### 7. Calculate the deterministic predictions. 

Calculate the deterministic predictions, deciding that it is healthy if the output probability is larger than 50%, in a variable `y_pred`. 

Plot the predictions for each data point, again superimposed on the results above using `ax.scatter()`. Use small red markers, using the arguments `c=tab:red` and `marker='.'`.

In [14]:
# Add your code below
y_pred = (y_prob > 0.5).astype(int)
ax.scatter(X, y_pred, c='tab:red', marker='.')
fig

#### 8. Notice that your model is deciding a sample is healthy if `ALS` is below a certain threhold. Calculate this threshold from the bias and weight parameters of the model. 
 
Draw the threshold as a vertical line, superimposed on the plots above using `ax.plot()`, and again in the color red. 

Note that this line represents a decision boundary: the model's decision is based on which side of it a sample lies.

*Hint: use the `model.intercept_` and `model.coef_` attributes as bias and weight attributes respectively, and that at the decision bourdary we have $w x - b = 0$, where $w$ is the weight and $b$ the bias.*

*Hint: to draw a vertical line at some value, `x`, use `ax.plot([x,x], [0,1])`.*

In [15]:
# Add your code below
w = model_AST.coef_[0]  
b = model_AST.intercept_
ALS_thres = -b / w 
ax.plot([ALS_thres[0], ALS_thres[0]], [0, 1], color='tab:red')
fig

#### 9. Calculate and plot the ROC curve for the predictions. 

ROC Curves summarise the trade-off between the true positive rate and false positive rate for a predictive model using different probability thresholds.

ROC curves are appropriate when the observations are balanced between each class, whereas precision-recall curves are appropriate for imbalanced datasets.

Use the new axis for the plot, using `ax_roc.plot()`. Limit the plot range to valid values, between zero and one.

In [16]:
from sklearn.metrics import roc_curve
fig_roc, ax_roc = plt.subplots()
# Add your code below

ROC = roc_curve(y, y_prob)
fpr, tpr, thresholds = ROC
ax_roc.plot(fpr, tpr)
ax_roc.set_xlim([0, 1])
ax_roc.set_ylim([0, 1])


## Analysis of multivariate logistic regression model

#### Now, we will use all the input variables.

Let's start by uncommenting the cell below and inspecting the first 3 lines of our `df_clean` DataFrame.

In [17]:
df_clean.head(3)

#### 10. Create a logistic regression model using all input columns from the clean data frame. 

#### Optimization algorithms can perform better with normalized input variables. To normalize the input features of your model (`X`), create a new variable called `X_norm` and use `X_norm` when training your model called `full_model`.

*Hint:  Use the z-score transformation $$\frac{x - \mu}{\sigma}$$ with the  `mean()` and `std()` methods.*

In [18]:
# Add your code below
X = df_clean.drop(columns=['healthy'])
print("Columns in X:", X.columns.tolist())
print("Data types:", X.dtypes)

# Convert to numeric, handling any non-numeric values
X_numeric = X.apply(pd.to_numeric, errors='coerce')
print("After conversion - any NaN:", X_numeric.isnull().sum().sum())

# Z-score normalization keeping DataFrame structure
X_mean = X_numeric.mean()
X_std = X_numeric.std()
# Avoid division by zero for constant columns
X_std = X_std.where(X_std != 0, 1)
X_norm = (X_numeric - X_mean) / X_std

# Convert to numpy array only for model training
y = df_clean['healthy'].values
full_model = logreg(X_norm.values, y)

#### 11. Calculate the prediction probabilities. 

#### Plot the histograms of the prediction probabilities, separately for healthy cases and for unhealthy cases. Do you think the model is any good?

Set the numbers of bins to 30 (`bins=30`), use the `step` histogram type (`histtype='step'`), and set the density mode to on (`density=True`).

In [19]:
# Add your code below
y_prob2 = full_model.predict_proba(X_norm.values)[:, 1]

plt.figure()
healthy_mask = y == 1
unhealthy_mask = y == 0
plt.hist(y_prob2[healthy_mask], bins=30, histtype='step', density=True, label='Healthy')
plt.hist(y_prob2[unhealthy_mask], bins=30, histtype='step', density=True, label='Unhealthy')
plt.legend()



#### The ROC curve indicate the false positive and negative error rates for different choices of decision thresholds, varying from zero to one.

#### 12. Calculate the ROC for the full model.

#### Superimpose the result over the ROC for the `ALS` model, reusing the `ax_roc` axis above. 

####  Which model has better predictions?

In [20]:
# Add your code below
ROC2 = roc_curve(y, y_prob2)
fpr2, tpr2, thresholds2 = ROC2
ax_roc.plot(fpr2, tpr2)
fig_roc

#### 13. Calculate the confusion matrix for the predictions using a 50% threshold. What's the error probability for healthy cases (false negatives) and unhealthy cases (false positives)?

You can review the elements of a confusion matrix in [link](https://www.nbshare.io/notebook/626706996/Learn-And-Code-Confusion-Matrix-With-Python/).

#### Notice that false positives and false negatives have very different probabilities. Think about why this is the case.

*Hint: Remember the imbalance in number of healthy and unhealthy samples.*

In [21]:
from sklearn.metrics import confusion_matrix
# Add your code below

y_pred_full = (y_prob2 > 0.5).astype(int)
CM = confusion_matrix(y, y_pred_full)

# sklearn confusion matrix format:
# [[TN, FP],
#  [FN, TP]]
# where rows are actual, columns are predicted

# Calculate error probabilities
tn, fp, fn, tp = CM.ravel()


# false_neg = error rate for unhealthy cases (FP rate)
# false_pos = error rate for healthy cases (FN rate)
false_neg = fp / (tn + fp)  
false_pos = fn / (tp + fn)  


Once you have implemented the confusion matrix and the error probabilities for healthy and unhealthy cases, uncomment and run the cell below to inspect these values:

In [22]:
print(CM)
print(false_neg)
print(false_pos)

#### 14. Calculate and print the decision probability threshold that would give same error probability for false positives and false negatives. 

*Hint: Find where in ROC curve false negatives become larger than false positives, using the `np.where()` function. Then use the third element of the ROC output (`ROC2[2]`), which tells the probability threshold at that point of the ROC curve.*

In [26]:
# # Add your code below
# fnr = 1 - tpr2  # FNR = 1 - True Positive Rate

# # Find the index where FPR and FNR are closest (balanced)
# # This finds where |FPR - FNR| is minimized
# index_roc = np.argmin(np.abs(fpr2 - fnr))

# # crossing_points = np.where(fpr2 > fnr)[0]
# # if len(crossing_points) > 0:
# #     index_roc = crossing_points[-1]  # Last point where fpr2 > fnr
# # else:
# #     # Fallback to balanced point
# #     index_roc = np.argmin(np.abs(fpr2 - fnr))

# # Get the threshold at this balanced point
# decision_thres = thresholds2[index_roc]

# print(f"Using crossing approach - index_roc: {index_roc}")
# print(f"At this index: FPR={fpr2[index_roc]:.4f}, FNR={fnr[index_roc]:.4f}")

# print("Debugging ROC curve indices:")
# print(f"Length of fpr2: {len(fpr2)}")
# print(f"Length of tpr2: {len(tpr2)}")
# print(f"Length of thresholds2: {len(thresholds2)}")

# # Calculate FNR
# fnr = 1 - tpr2

# # Let's check values around the expected index
# for i in range(15, 25):  # Check around index 19
#     if i < len(fpr2):
#         print(f"Index {i}: FPR={fpr2[i]:.4f}, FNR={fnr[i]:.4f}, diff={abs(fpr2[i] - fnr[i]):.4f}, threshold={thresholds2[i]:.4f}")

# # Calculate the balanced point
# diff = np.abs(fpr2 - fnr)
# index_roc = np.argmin(diff)
# decision_thres = thresholds2[index_roc]

# print(f"\nCalculated index_roc: {index_roc}")
# print(f"Decision threshold: {decision_thres}")


# Add your code below
fnr = 1 - tpr2  # FNR = 1 - True Positive Rate

# Find where FNR transitions from being less than FPR to greater than FPR
diff = fpr2 - fnr
sign_changes = np.where(np.diff(np.sign(diff)))[0]

if len(sign_changes) > 0:
    # Use the point just after the sign change
    index_roc = sign_changes[0] + 1
else:
    # Fallback to balanced point
    index_roc = np.argmin(np.abs(fpr2 - fnr))

# Get the threshold at this balanced point
decision_thres = thresholds2[index_roc]

# # Add your code below
# fnr = 1 - tpr2  # FNR = 1 - True Positive Rate

# # Let's debug by looking at the actual values
# # Print some values around index 19 to understand the pattern
# for i in range(0, 25):
#     if i < len(fpr2):
#         print(f"Index {i}: FPR={fpr2[i]:.4f}, FNR={fnr[i]:.4f}, diff={fpr2[i] - fnr[i]:.4f}")

# # The hint says "find where false negatives become larger than false positives"
# # This means find where FNR > FPR
# # But we want the point just before this transition, or the exact crossing

# # Method 1: Find where FNR first becomes larger than FPR
# crossing_indices = np.where(fnr > fpr2)[0]
# if len(crossing_indices) > 0:
#     # Take the first index where FNR > FPR
#     index_roc = crossing_indices[0]
# else:
#     # Fallback
#     index_roc = np.argmin(np.abs(fpr2 - fnr))

# # If that doesn't work, try method 2: Find the transition point
# if index_roc != 19:  # If we didn't get the expected result
#     # Look for sign changes in (FPR - FNR)
#     diff = fpr2 - fnr
#     # Find where diff changes from positive to negative (FPR stops being > FNR)
#     for i in range(1, len(diff)):
#         if diff[i-1] > 0 and diff[i] <= 0:
#             index_roc = i
#             break

# decision_thres = thresholds2[index_roc]
# print(f"Final index_roc: {index_roc}")
# print(f"Decision threshold: {decision_thres}")

When you have calculated `decision_thres`, uncomment the cell below to print it out:

In [27]:
decision_thres

In [32]:
# Let's verify we're using the right ROC curve data
print("Checking ROC2 components:")
print(f"Type of ROC2: {type(ROC2)}")
print(f"ROC2 contents: {ROC2}")
print(f"fpr2 first 5 values: {fpr2[:5]}")
print(f"tpr2 first 5 values: {tpr2[:5]}")
print(f"thresholds2 first 5 values: {thresholds2[:5]}")

# Check if ROC2[2] gives us the same thresholds
print(f"ROC2[2] first 5 values: {ROC2[2][:5]}")
print(f"Are thresholds2 and ROC2[2] the same? {np.array_equal(thresholds2, ROC2[2])}")

#### 15. Mark over the ROC curve the point corresponding to the balanced error choice above.

#### Use `ax_roc.plot()` to plot over the ROC figure above. Plot a single marker by plotting the x and y values of the ROC curve in `ROC2` for the correct index. Use the plotting argument `gx` for a green X marker.

#### Also, calculate the ROC point for the 50% decision threshold considered above. Mark it as a red X marker in the same figure.

#### Which decision threshold is preferable? What would this preference depend on?

*Hint: Think about what is worse, false positive or false negatives?*

In [26]:
# Add your code below
# Assign the calculated index to ROC_index
ROC_index = index_roc

# Plot the balanced error point (green X marker)
ax_roc.plot(fpr2[ROC_index], tpr2[ROC_index], 'gx', markersize=10, markeredgewidth=3)

# Calculate and plot the 50% threshold point (red X marker)
idx_50 = np.argmin(np.abs(thresholds2 - 0.5))
ax_roc.plot(fpr2[idx_50], tpr2[idx_50], 'rx', markersize=10, markeredgewidth=3)
fig_roc

## Implementing the model from scratch

Most machine learning models are trained with gradient descent, updating model parameters with small changes in the direction that reduces the training loss. 

#### To learn in detail what goes on when we use libraries to train our model,  we will now implement the gradient descent algorithm from scratch and compare the results with the scikit-learn package model. 

#### We give you here the gradient and loss functions for the logistic regression model.

In [27]:
sigma = lambda u: 1/(1+np.exp(-u))


def gradient(w, x, y):
    y_prob = sigma(x@w)
    return np.dot(x.T, (y_prob - y))/len(y)

def loss(w, x, y):
    y_prob = sigma(x@w)
    # Clip probabilities to avoid log(0) and log(1)
    epsilon = 1e-15
    y_prob = np.clip(y_prob, epsilon, 1 - epsilon)
    return -(y*np.log(y_prob)+(1-y)*np.log(1-y_prob)).mean()

#### 16. Implement a function that takes the input and output of the logistic regression model and fits it using gradient descent. 

Use random Gaussian initialization (with unit variance), no regularization, learning rate of 2., and 200 gradient descent steps. 

Also, calculate the total loss at each step. Return the final weight parameters and the total loss time series.

In [ ]:
np.random.seed(0)
# Add your code below


def fit(X, y):
    # Add bias column at the END for intercept
    X_with_bias = np.column_stack([X, np.ones(X.shape[0])])
    
    # Random Gaussian initialization with unit variance
    w = np.random.normal(0, 1, X_with_bias.shape[1])
    
    # Training parameters: learning rate 2.0, 200 steps
    learning_rate = 2.0
    n_steps = 200
    loss_t = []
    
    # Gradient descent loop
    for t in range(n_steps):
        grad = gradient(w, X_with_bias, y)
        w = w - learning_rate * grad
        
        # Calculate loss AFTER the gradient update
        current_loss = loss(w, X_with_bias, y)
        loss_t.append(current_loss)
    
    return w, loss_t


#### 17. Fit the model to our clean and normalized inputs. 

#### Calculate and print the final weights, named `w`, and plot the loss over time, named `loss_t`. 

#### Does it learn over time? Has it nearly converged?

Remember to add a constant column to the input for the bias parameter.

In [ ]:
# Add your code below
X_values = X_norm.values.astype(float)  # Convert to numpy for custom implementation
Y = df_clean['healthy'].values  # Using Y as in instructions

w, loss_t = fit(X_values, Y)
print("Final weights:", w)
plt.figure()
plt.plot(loss_t)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Loss over time')


#### 18. Calculate and plot the probability predictions histograms for healthy and unhealthy samples. 

#### Use the same histogram properties as above

#### Has this model learned well?

*Hint: the output probability of the logistic regression model is $p(x=1) = \sigma(w^T x-b)$*.

In [ ]:
# Add your code below
X_with_bias = np.column_stack([X_values, np.ones(X_values.shape[0])])  # Same order as in fit()
y_prob3 = sigma(X_with_bias @ w)

plt.figure()
healthy_mask = Y == 1
unhealthy_mask = Y == 0
plt.hist(y_prob3[healthy_mask], bins=30, histtype='step', density=True, label='Healthy')
plt.hist(y_prob3[unhealthy_mask], bins=30, histtype='step', density=True, label='Unhealthy')
plt.legend()

#### 19. Compare the coefficients learned with our model and the scikit-learn model above. 

#### Make a scatter plot  using `plt.scatter`, with the vector of coefficients of each of the model as arguments. 

#### As a reference, plot a black dashed line on the main diagonal of the plot (i.e. where x=y). 

*Hint: Use `plt.plot([a,a],[b,b])` for some values `a` and `b`, and `k--` as argument.* 

#### Also, calculate and print the correlation between these two vectors, using the method `np.corrcoef()`, named `correlation`. 

#### Are the two models similar? Why should they be different, similar or identical?

In [ ]:
# Add your code below
sklearn_coefs = np.concatenate([full_model.coef_[0], full_model.intercept_])  # Features first, then intercept
custom_coefs = w

plt.figure()
plt.scatter(sklearn_coefs, custom_coefs)

# Plot diagonal reference line as specified
min_val = min(sklearn_coefs.min(), custom_coefs.min())
max_val = max(sklearn_coefs.max(), custom_coefs.max())
plt.plot([min_val, max_val], [min_val, max_val], 'k--')

plt.xlabel('Sklearn Coefficients')
plt.ylabel('Custom Coefficients')
plt.title('Coefficient Comparison')

# Calculate correlation
correlation = np.corrcoef(sklearn_coefs, custom_coefs)[0, 1]
print(f"Correlation: {correlation:.4f}")